In [1]:
def extract_qza(qza_path, extract_to):
    import zipfile
    import os

    # Create output folder if it doesn't exist
    os.makedirs(extract_to, exist_ok=True)

    # Extract contents
    with zipfile.ZipFile(qza_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)

    print(f"Extracted contents to: {extract_to}")

extract_qza(
    "D:/Anil/self/micro_metabolite/data/raw_data/ankita/004_picrust_ankita/taxonomy_ankita_08_12_25.qza",
    "D:/Anil/self/micro_metabolite/data/raw_data/ankita/004_picrust_ankita"
)
extract_qza(
    "D:/Anil/self/micro_metabolite/data/raw_data/arko/004_picrust_arko/taxonomy_arko_08_12_25.qza",
    "D:/Anil/self/micro_metabolite/data/raw_data/arko/004_picrust_arko"
)
extract_qza(
    "D:/Anil/self/micro_metabolite/data/raw_data/tanuja/004_picrust_tanuja/taxonomy_tanuja_12_12_25.qza",
    "D:/Anil/self/micro_metabolite/data/raw_data/tanuja/004_picrust_tanuja"
)

Extracted contents to: D:/Anil/self/micro_metabolite/data/raw_data/ankita/004_picrust_ankita
Extracted contents to: D:/Anil/self/micro_metabolite/data/raw_data/arko/004_picrust_arko
Extracted contents to: D:/Anil/self/micro_metabolite/data/raw_data/tanuja/004_picrust_tanuja


In [ ]:
# Read the CSV files
def process_picrust_data(taxonomy_file, path_abun_file, cleaned_output):
    import pandas as pd

    A = pd.read_csv(taxonomy_file, sep="\t")
    B = pd.read_csv(path_abun_file, sep="\t")
    A_sel = A.iloc[:, [0, 1]]
    B_sel = B.iloc[:, [0, 1, 2, 7]]
    A_sel[A_sel.columns[0]] = A_sel[A_sel.columns[0]].astype(str)
    B_sel[B_sel.columns[2]] = B_sel[B_sel.columns[2]].astype(str)
    df = pd.merge(
        A_sel, B_sel, left_on=A_sel.columns[0], right_on=B_sel.columns[2], how="inner"
    )
    df.drop(df.columns[[0, 4]], axis=1, inplace=True)
    df = df.groupby([df.columns[0], df.columns[1], df.columns[2]], as_index=False).sum()
    df.to_csv(cleaned_output, index=False)
    print("✅ Cleaned file saved successfully.")


process_picrust_data(
    "D:/Anil/self/micro_metabolite/data/raw_data/ankita/004_picrust_ankita/dd0c9176-4a6a-490f-b99e-7a36dde74004/data/taxonomy.tsv",
    "D:/Anil/self/micro_metabolite/data/raw_data/ankita/004_picrust_ankita/pred_metagenome_contrib_ankita_08_12_25.tsv.gz",
    "D:/Anil/self/micro_metabolite/data/raw_data/ankita/004_picrust_ankita/picrust_ko_ankita.csv"
)

process_picrust_data(
    "D:/Anil/self/micro_metabolite/data/raw_data/arko/004_picrust_arko/0252992c-201a-409c-b03a-360109f0ea73/data/taxonomy.tsv",
    "D:/Anil/self/micro_metabolite/data/raw_data/arko/004_picrust_arko/pred_metagenome_contrib_arko_08_12_25.tsv.gz",
    "D:/Anil/self/micro_metabolite/data/raw_data/arko/004_picrust_arko/picrust_ko_arko.csv"
)

process_picrust_data(
    "D:/Anil/self/micro_metabolite/data/raw_data/tanuja/004_picrust_tanuja/f89d53ec-a2b8-4f7f-aa26-c6b3b58b92f2/data/taxonomy.tsv",
    "D:/Anil/self/micro_metabolite/data/raw_data/tanuja/004_picrust_tanuja/pred_metagenome_contrib_tanuja_12_12_25.tsv.gz",
    "D:/Anil/self/micro_metabolite/data/raw_data/tanuja/004_picrust_tanuja/picrust_ko_tanuja.csv"
)

In [4]:
def merge_picrust_metadata(picrust_path, metadata_path, output_path):
    import pandas as pd

    """
    Merge picrust and metadata files.
    """
    print("=" * 50)
    print("Starting merge process...")
    print("=" * 50)

    # Read files
    print("\n[1/6] Reading CSV file...")
    picrust = pd.read_csv(picrust_path)
    print(
        f"      ✓ Picrust loaded: {picrust.shape[0]} rows, {picrust.shape[1]} columns"
    )

    print("\n[2/6] Reading Excel file...")
    metadata = pd.read_excel(metadata_path)
    print(
        f"      ✓ Metadata loaded: {metadata.shape[0]} rows, {metadata.shape[1]} columns"
    )

    # Merge on 'sample' column
    print("\n[3/6] Merging files on 'sample' column...")
    merge_df = pd.merge(picrust, metadata, on="sample")
    print(
        f"      ✓ Merged successfully: {merge_df.shape[0]} rows, {merge_df.shape[1]} columns"
    )

    # Drop 'sample' column
    print("\n[4/6] Dropping 'sample' column and renaming 'sampleID' to 'sample'...")
    merged_df = merge_df.drop(columns=["sample"])
    merged_df = merged_df.rename(columns={"sampleID": "sample"})
    print(
        f"      ✓ Final dataframe: {merged_df.shape[0]} rows, {merged_df.shape[1]} columns"
    )

    # Move 'sample' column to second position (after first column)
    print("\n[5/6] Moving 'sample' column to second position...")
    sample_col = merged_df.pop("sample")  # Remove sample column
    merged_df.insert(1, "sample", sample_col)  # Insert at position 1 (second column)
    print(
        f"      ✓ Column order updated: {list(merged_df.columns[:5])}..."
    )  # Show first 5 columns

    # Save to file
    print("\n[6/6] Saving to tab-separated text file...")
    merged_df.to_csv(output_path, sep="\t", index=False)
    print(f"      ✓ File saved to: {output_path}")

    print("\n" + "=" * 50)
    print("Process completed successfully!")
    print("=" * 50)

    return merged_df


# Call the function
merge_picrust_metadata(
    picrust_path="D:/Anil/self/micro_metabolite/data/raw_data/ankita/004_picrust_ankita/picrust_ko_ankita.csv",
    metadata_path="D:/Anil/self/micro_metabolite/data/raw_data/ankita/004_picrust_ankita/metadata_ankita.xlsx",
    output_path="D:/Anil/self/micro_metabolite/data/raw_data/picrust/picrust_ko_ankita_edited.txt"
)
merge_picrust_metadata(
    picrust_path="D:/Anil/self/micro_metabolite/data/raw_data/arko/004_picrust_arko/picrust_ko_arko.csv",
    metadata_path="D:/Anil/self/micro_metabolite/data/raw_data/arko/004_picrust_arko/metadata_arko.xlsx",
    output_path="D:/Anil/self/micro_metabolite/data/raw_data/picrust/picrust_ko_arko_edited.txt"
)
merge_picrust_metadata(
    picrust_path="D:/Anil/self/micro_metabolite/data/raw_data/tanuja/004_picrust_tanuja/picrust_ko_tanuja.csv",
    metadata_path="D:/Anil/self/micro_metabolite/data/raw_data/tanuja/004_picrust_tanuja/metadata_tanuja.xlsx",
    output_path="D:/Anil/self/micro_metabolite/data/raw_data/picrust/picrust_ko_tanuja_edited.txt"
)

Starting merge process...

[1/6] Reading CSV file...
      ✓ Picrust loaded: 276767 rows, 4 columns

[2/6] Reading Excel file...
      ✓ Metadata loaded: 26 rows, 2 columns

[3/6] Merging files on 'sample' column...
      ✓ Merged successfully: 276767 rows, 5 columns

[4/6] Dropping 'sample' column and renaming 'sampleID' to 'sample'...
      ✓ Final dataframe: 276767 rows, 4 columns

[5/6] Moving 'sample' column to second position...
      ✓ Column order updated: ['Taxon', 'sample', 'function', 'taxon_rel_function_abun']...

[6/6] Saving to tab-separated text file...
      ✓ File saved to: D:/Anil/self/micro_metabolite/data/raw_data/picrust/picrust_ko_ankita_edited.txt

Process completed successfully!
Starting merge process...

[1/6] Reading CSV file...
      ✓ Picrust loaded: 4190541 rows, 4 columns

[2/6] Reading Excel file...
      ✓ Metadata loaded: 25 rows, 2 columns

[3/6] Merging files on 'sample' column...
      ✓ Merged successfully: 4190541 rows, 5 columns

[4/6] Dropping 's

,Taxon,sample,function,taxon_rel_function_abun
0,Unassigned,N4CST1,K00001,8.770543
1,Unassigned,N4CST1,K00003,11.834131
2,Unassigned,N4CST1,K00005,2.711812
3,Unassigned,N4CST1,K00008,10.550510
4,Unassigned,N4CST1,K00009,0.394322
...,...,...,...,...
2004743,d__Bacteria; p__Verrucomicrobiota; c__Verrucom...,W24ST2,K19265,0.003146
2004744,d__Bacteria; p__Verrucomicrobiota; c__Verrucom...,W24ST2,K19271,0.003146
2004745,d__Bacteria; p__Verrucomicrobiota; c__Verrucom...,W24ST2,K19405,0.003146
2004746,d__Bacteria; p__Verrucomicrobiota; c__Verrucom...,W24ST2,K19411,0.003146


In [ ]:
def process_excel(input_path, output_path, columns_to_drop):
    import pandas as pd
    """
    Read Excel, drop specified columns, save as tab-separated txt.
    
    Parameters:
    -----------
    input_path : str
        Path to input Excel file
    output_path : str
        Path for output tab-separated text file
    columns_to_drop : list
        List of column indices to drop
    """
    print("=" * 50)
    print("Starting process...")
    print("=" * 50)
    
    # Read Excel file
    print("\n[1/3] Reading Excel file...")
    df = pd.read_csv(input_path)
    print(f"      ✓ Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"      ✓ Columns: {list(df.columns)}")
    
    # Drop columns by index
    print(f"\n[2/3] Dropping columns at index {columns_to_drop}...")
    dropped_cols = [df.columns[i] for i in columns_to_drop]
    df = df.drop(df.columns[columns_to_drop], axis=1)
    print(f"      ✓ Dropped columns: {dropped_cols}")
    print(f"      ✓ Remaining: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # Save as tab-separated text file
    print("\n[3/3] Saving to tab-separated text file...")
    df.to_csv(output_path, sep='\t', index=False)
    print(f"      ✓ File saved to: {output_path}")
    
    print("\n" + "=" * 50)
    print("Process completed successfully!")
    print("=" * 50)
    
    return df


# Call the function
process_excel(
    input_path="D:/Anil/self/micro_metabolite/data/raw_data/picrust/picrust_ko_old.csv",
    output_path="D:/Anil/self/micro_metabolite/data/raw_data/picrust/picrust_ko_old_edited.txt",
    columns_to_drop=[0, 4]
);

In [5]:
def stack_txt_files(folder_path, output_path):
    import pandas as pd
    import glob
    import os
    """
    Stack all tab-separated txt files from a folder.
    
    Parameters:
    -----------
    folder_path : str
        Path to folder containing txt files
    output_path : str
        Path for output stacked file
    """
    print("=" * 50)
    print("Starting stacking process...")
    print("=" * 50)
    
    # Get all txt files
    print("\n[1/3] Finding all txt files...")
    all_files = glob.glob(os.path.join(folder_path, "*.txt"))
    print(f"      ✓ Found {len(all_files)} files")
    
    # Read and stack all files
    print("\n[2/3] Reading and stacking files...")
    df_list = []
    for file in all_files:
        df = pd.read_csv(file, sep='\t')
        df_list.append(df)
        print(f"      ✓ {os.path.basename(file)}: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # Concatenate all dataframes
    stacked_df = pd.concat(df_list, ignore_index=True)
    print(f"\n      ✓ Total stacked: {stacked_df.shape[0]} rows, {stacked_df.shape[1]} columns")
    
    # Save to tab-separated txt file
    print("\n[3/3] Saving stacked file...")
    stacked_df.to_csv(output_path, sep='\t', index=False)
    print(f"      ✓ File saved to: {output_path}")
    
    print("\n" + "=" * 50)
    print("Process completed successfully!")
    print("=" * 50)
    
    return stacked_df


# Call the function
stack_txt_files(
    folder_path="D:/Anil/self/micro_metabolite/data/raw_data/picrust/",
    output_path="D:/Anil/self/micro_metabolite/data/raw_data/picrust/picrust_ko_all_stacked.txt"
);

Starting stacking process...

[1/3] Finding all txt files...
      ✓ Found 4 files

[2/3] Reading and stacking files...
      ✓ picrust_ko_ankita_edited.txt: 276767 rows, 4 columns
      ✓ picrust_ko_arko_edited.txt: 4190541 rows, 4 columns
      ✓ picrust_ko_old_edited.txt: 16720537 rows, 4 columns
      ✓ picrust_ko_tanuja_edited.txt: 2004748 rows, 4 columns

      ✓ Total stacked: 23192593 rows, 4 columns

[3/3] Saving stacked file...
      ✓ File saved to: D:/Anil/self/micro_metabolite/data/raw_data/picrust/picrust_ko_all_stacked.txt

Process completed successfully!
